# Sentiment Analysis using NLP Pipeline & ML Models

In [28]:
import pandas as pd
import numpy as np

In [29]:
df = pd.read_csv("/kaggle/input/datasets/shreyasutar14/imdb-reviews/IMDB Dataset.csv")
df = df.sample(10000)
df.head()

,review,sentiment
18685,Splendid film that in just eight minutes displ...,positive
11412,Is this movie as bad as some claim? In my opin...,negative
291,"I had some reservations about this movie, I fi...",positive
47571,"""A Classic is something that everybody wants t...",positive
20150,"Slick, tidy, and well-made old school version ...",positive


In [30]:
df.shape

(10000, 2)

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 18685 to 10940
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     10000 non-null  object
 1   sentiment  10000 non-null  object
dtypes: object(2)
memory usage: 234.4+ KB


In [32]:
df['sentiment'].value_counts()

sentiment
negative    5009
positive    4991
Name: count, dtype: int64

## NLP Preprocessing

In [33]:
!pip install nltk

In [36]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [37]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [39]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]
    
    ps = PorterStemmer()
    words = [ps.stem(word) for word in words]
    
    return " ".join(words)

In [40]:
df['clean_review'] = df['review'].apply(clean_text)

In [41]:
df[['review', 'clean_review']].head()

,review,clean_review
18685,Splendid film that in just eight minutes displ...,splendid film eight minut display unusu genr m...
11412,Is this movie as bad as some claim? In my opin...,movi bad claim opinion ye go comment note quit...
291,"I had some reservations about this movie, I fi...",reserv movi figur would usual bill fare formul...
47571,"""A Classic is something that everybody wants t...",classic someth everybodi want read nobodi want...
20150,"Slick, tidy, and well-made old school version ...",slick tidi well made old school version great ...


## Feature Engineering

In [42]:
!pip install scikit-learn

In [43]:
from sklearn.feature_extraction.text import CountVectorizer

In [44]:
cv = CountVectorizer(max_features=5000)
X_bow = cv.fit_transform(df['clean_review']).toarray()

In [45]:
X_bow.shape

(10000, 5000)

In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [48]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_review']).toarray()

In [49]:
X_tfidf.shape

(10000, 5000)

In [50]:
from sklearn.model_selection import train_test_split

X = X_tfidf   
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Model Building

In [51]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [52]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

In [54]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=10)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

## Model Evaluation

In [55]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [56]:
print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr, pos_label='positive'))
print("Recall:", recall_score(y_test, y_pred_lr, pos_label='positive'))
print("F1 Score:", f1_score(y_test, y_pred_lr, pos_label='positive'))

Logistic Regression
Accuracy: 0.869
Precision: 0.8568646543330087
Recall: 0.8844221105527639
F1 Score: 0.8704253214638972


In [57]:
print("\nNaive Bayes")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("Precision:", precision_score(y_test, y_pred_nb, pos_label='positive'))
print("Recall:", recall_score(y_test, y_pred_nb, pos_label='positive'))
print("F1 Score:", f1_score(y_test, y_pred_nb, pos_label='positive'))


Naive Bayes
Accuracy: 0.8415
Precision: 0.8410462776659959
Recall: 0.8402010050251256
F1 Score: 0.840623428858723


In [58]:
print("\nDecision Tree")
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt, pos_label='positive'))
print("Recall:", recall_score(y_test, y_pred_dt, pos_label='positive'))
print("F1 Score:", f1_score(y_test, y_pred_dt, pos_label='positive'))


Decision Tree
Accuracy: 0.7195
Precision: 0.6767100977198697
Recall: 0.8351758793969849
F1 Score: 0.747638326585695


## Comparison & Insights

Three models were trained: Logistic Regression, Naive Bayes, and Decision Tree. Their performance was evaluated using accuracy, precision, recall, and F1 score.

Logistic Regression performed best due to its ability to handle high-dimensional sparse data efficiently. Naive Bayes performed well but assumes feature independence. Decision Tree showed lower performance due to overfitting on text data.

## Summary of Findings

- The dataset was preprocessed using NLP techniques such as lowercasing, stopword removal, and stemming.
- Feature extraction was performed using Bag of Words and TF-IDF.
- Three models were trained: Logistic Regression, Naive Bayes, and Decision Tree.
- Logistic Regression achieved the best performance among all models.
- Naive Bayes performed well but assumes feature independence.
- Decision Tree showed lower performance due to overfitting.

Overall, TF-IDF with Logistic Regression provided the best results for sentiment classification.